# Notebook 01: Extração e Pré-processamento
Este notebook filtra os idosos do banco mestre e cria os 4 subgrupos clínicos.

In [1]:
# Importa o módulo sqlite3, que permite conversar com bancos de dados SQLite locais.
import sqlite3
# Importa o pandas, a biblioteca principal para manipulação de tabelas (DataFrames) no Python.
import pandas as pd
# Importa o módulo os, que nos permite lidar com caminhos de pastas e arquivos no Windows.
import os

# Define o caminho onde os bancos de dados estão salvos.
PASTA_DB = '../data/database/'
# Junta o caminho da pasta com o nome do arquivo do banco mestre gigante.
DB_MASTER = os.path.join(PASTA_DB, 'pns_master_formatado.db')

# Imprime na tela um aviso para ver se o arquivo de fato existe, retornando True ou False.
print(f"Verificando existência do banco mestre: {os.path.exists(DB_MASTER)}")


Verificando existência do banco mestre: True


In [2]:
# Cria uma lista com o código das 14 perguntas sobre doenças crônicas do questionário da PNS (Módulo Q).
# O Q079 é a variável principal do nosso estudo: diagnóstico de Artrite ou Reumatismo.
variaveis_doencas = [
    'Q00201', # Hipertensão
    'Q03001', # Diabetes
    'Q060',   # Colesterol
    'Q06306', # Doença do coração
    'Q068',   # AVC
    'Q074',   # Asma
    'Q079',   # Artrite/Reumatismo (Nossa Variável Alvo)
    'Q088',   # DORT
    'Q092',   # Depressão
    'Q11006', # Outra doença mental
    'Q11604', # Doença crônica no pulmão
    'Q120',   # Câncer
    'Q124',   # Insuficiência renal crônica
    'Q128'    # Outra doença crônica
]

# Cria um comando (string) SQL para buscar apenas as pessoas com 60 anos ou mais.
# Como a idade (C008) está salva como texto no banco, usamos CAST para forçar o SQLite a ler como número inteiro (INTEGER).
filtro_idosos = "CAST(C008 AS INTEGER) >= 60"


In [3]:
# Avisa o usuário que a extração vai começar (pode demorar alguns segundos).
print("Extraindo Idosos Geral...")

# Abre uma conexão direta com o arquivo do banco mestre gigante.
conn_master = sqlite3.connect(DB_MASTER)

# Monta a query SQL: Selecione TODAS as colunas (*) da tabela 'pns_completa' ONDE a idade for >= 60.
query_geral = f"SELECT * FROM pns_completa WHERE {filtro_idosos}"

# O Pandas vai até o banco, executa a query e traz a tabela de idosos direto para a memória RAM.
df_idosos = pd.read_sql_query(query_geral, conn_master)

# Fecha a conexão com o banco mestre para não travar o arquivo.
conn_master.close()

# Imprime o número exato de linhas (idosos) que vieram da pesquisa.
print(f"Total de Idosos encontrados: {len(df_idosos)}")

# Cria um NOVO arquivo de banco de dados chamado 'idosos_geral.db' na pasta data/database/.
conn_geral = sqlite3.connect(os.path.join(PASTA_DB, 'idosos_geral.db'))

# Joga toda a tabela de idosos (df_idosos) para dentro desse novo banco, criando uma tabela chamada 'pns_idosos'.
# O comando if_exists='replace' garante que se você rodar de novo, ele apaga o velho e salva o novo.
df_idosos.to_sql('pns_idosos', conn_geral, if_exists='replace', index=False)

# Fecha a conexão do banco novo.
conn_geral.close()

# Avisa que deu tudo certo.
print("Banco 'idosos_geral.db' salvo com sucesso.")


Extraindo Idosos Geral...
Total de Idosos encontrados: 43554
Banco 'idosos_geral.db' salvo com sucesso.


## Critério de classificação Sim/Não (diagnóstico autorreferido)

As 14 variáveis do Módulo Q (doenças crônicas) assumem no banco apenas **`Sim`**, **`Não`** ou
**`NULL`** (não-aplicável / morador não selecionado) — não há `Não sabe`/`Ignorado` como texto.

**Convenção adotada:**
- **Artrite** = `Q079 = Sim` (grupo de casos).
- **Saudável** exige **`Não` explícito nas 14** doenças. `NULL` (ou qualquer valor que não seja
  exatamente `Não`) **não** conta como `Não`, então esses moradores **não** entram em Saudável
  (critério conservador).
- **Limitação p/ o artigo:** diagnóstico **autorreferido** — ausência de afirmação de diagnóstico
  é tratada como ausência da doença.


In [4]:
# == Classificação Sim/Não das variáveis de doença (módulo Q) ====================
# No banco formatado os valores são SOMENTE: 'Sim', 'Não' ou NULL (não-aplicável /
# morador não selecionado). NÃO há 'Não sabe'/'Ignorado' como texto (verificado nos dados).
# CRITÉRIO (diagnóstico autorreferido):
#   - eh_sim -> apenas 'Sim' (ou código 1) = diagnóstico presente.
#   - eh_nao -> apenas 'Não' (ou código 2) = ausência. NULL e qualquer outro valor NÃO
#     contam como 'Não' -> não entram no grupo Saudável (que exige 'Não' nas 14). Conservador.
def eh_nao(val):
    if pd.isna(val): return False                          # NULL não é "Não"
    return str(val).strip().lower() in ('não', 'nao', '2')  # só 'Não' explícito (ou código 2)

def eh_sim(val):
    if pd.isna(val): return False
    return str(val).strip().lower() in ('sim', '1')         # só 'Sim' explícito (ou código 1)

# 1. Filtro da Artrite: Cria uma "máscara" (lista de Verdadeiro/Falso) dizendo quem tem artrite (Q079).
mask_artrite = df_idosos['Q079'].apply(eh_sim)

# 2. Filtro de Outras Doenças: Cria uma lista com o nome de todas as 13 doenças, excluindo a Artrite.
doencas_sem_artrite = [v for v in variaveis_doencas if v != 'Q079']

# Verifica se a pessoa respondeu SIM para QUALQUER UMA (.any(axis=1)) das 13 outras doenças.
try:
    mask_outras_doencas = df_idosos[doencas_sem_artrite].map(eh_sim).any(axis=1) # No pandas novo usa .map
except AttributeError:
    mask_outras_doencas = df_idosos[doencas_sem_artrite].applymap(eh_sim).any(axis=1) # No pandas velho usa .applymap

# 3. Filtro de Saudáveis: Verifica se a pessoa respondeu NÃO para ABSOLUTAMENTE TODAS (.all(axis=1)) as 14 doenças.
try:
    mask_saudavel = df_idosos[variaveis_doencas].map(eh_nao).all(axis=1)
except AttributeError:
    mask_saudavel = df_idosos[variaveis_doencas].applymap(eh_nao).all(axis=1)

# Aplica a máscara: separa apenas quem tem artrite (Com comorbidades)
df_artrite = df_idosos[mask_artrite]

# Aplica a máscara dupla: tem artrite (mask_artrite) E (símbolo &) NÃO TEM (~ símbolo de negação) outras doenças.
df_artrite_puro = df_idosos[mask_artrite & ~mask_outras_doencas]

# Aplica a máscara: separa os idosos 100% saudáveis.
df_saudaveis = df_idosos[mask_saudavel]

# Imprime o tamanho de cada subgrupo criado para você auditar.
print(f"Idosos com Artrite: {len(df_artrite)}")
print(f"Idosos com Artrite Pura: {len(df_artrite_puro)}")
print(f"Idosos Saudáveis: {len(df_saudaveis)}")


Idosos com Artrite: 4025
Idosos com Artrite Pura: 494
Idosos Saudáveis: 4332


In [5]:
# Abre o arquivo idosos_artrite.db e salva o grupo que tem Artrite.
conn_artrite = sqlite3.connect(os.path.join(PASTA_DB, 'idosos_artrite.db'))
df_artrite.to_sql('pns_idosos', conn_artrite, if_exists='replace', index=False) # if_exists='replace' sobrescreve
conn_artrite.close()

# Abre o arquivo idosos_artrite_puro.db e salva o grupo purificado.
conn_artrite_puro = sqlite3.connect(os.path.join(PASTA_DB, 'idosos_artrite_puro.db'))
df_artrite_puro.to_sql('pns_idosos', conn_artrite_puro, if_exists='replace', index=False)
conn_artrite_puro.close()

# Abre o arquivo idosos_saudaveis.db e salva os idosos saudáveis.
conn_saudaveis = sqlite3.connect(os.path.join(PASTA_DB, 'idosos_saudaveis.db'))
df_saudaveis.to_sql('pns_idosos', conn_saudaveis, if_exists='replace', index=False)
conn_saudaveis.close()

# Imprime o aviso final da etapa de extração!
print("Todos os bancos foram recriados e salvos com sucesso!")


Todos os bancos foram recriados e salvos com sucesso!
